# Exam No. 1 - Python Programming

Notebook này trình bày lời giải theo từng đoạn nhỏ để dễ chạy, dễ giải thích và dễ trình bày.

Nội dung chính:
- Đọc dữ liệu từ `gradedata_v4.csv`
- Khám phá dữ liệu
- Tạo biến mới `gender_num`, `pretest`, `posttest`
- Thống kê, trực quan hóa, tương quan
- Xây dựng mô hình dự đoán `pretest`
- So sánh `pretest` và `posttest` bằng Confusion Matrix


## 1. Chuẩn bị thư viện và cấu hình

Ghi chú:
- Dataset đang dùng là `D:\gradedata_v4.csv`
- Trong dữ liệu có tên cột `postest2`, `postest3` đúng theo file gốc, nên ta giữ nguyên tên đó khi xử lý


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import confusion_matrix, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_PATH = Path(r"D:\gradedata_v4.csv")
OUTPUT_DIR = Path("exam_no1_notebook_output")
PLOT_DIR = OUTPUT_DIR / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ["Low", "Medium", "Good", "Excellent"]


def save_fig(filename: str):
    plt.tight_layout()
    plt.savefig(PLOT_DIR / filename, dpi=200, bbox_inches="tight")
    plt.show()


## 2. Đọc dữ liệu

Bước này tương ứng với yêu cầu 1 của đề: nạp dữ liệu từ file CSV.


In [ ]:
df = pd.read_csv(DATA_PATH)

print("Đã đọc dữ liệu từ:", DATA_PATH)
print("Kích thước ban đầu:", df.shape)
display(df.head())


## 3. Số dòng, số cột, tên cột và 20 bản ghi đầu

Bước này giải quyết các ý 2, 3 và 4.


In [ ]:
print("Số dòng   :", df.shape[0])
print("Số cột    :", df.shape[1])

summary = pd.DataFrame(
    {
        "column_name": df.columns,
        "non_null_entries": [df[col].count() for col in df.columns],
        "data_type": [str(df[col].dtype) for col in df.columns],
    }
)

display(summary)
display(df.head(20))


## 4. Vẽ histogram cho các cột số

Bước này giải quyết ý 5: xem phân phối của các biến dạng số.


In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
print("Các cột số:", numeric_columns)

fig, axes = plt.subplots(4, 3, figsize=(18, 16))
axes = axes.flatten()

for ax, col in zip(axes, numeric_columns):
    sns.histplot(df[col], bins=15, kde=True, ax=ax, color="#2a9d8f")
    ax.set_title(col)

for ax in axes[len(numeric_columns):]:
    ax.axis("off")

plt.suptitle("Histogram cho các cột số", fontsize=16, y=1.02)
save_fig("01_histograms_numeric_columns.png")


## 5. Groupby theo gender, age và gender-age

Bước này giải quyết ý 6.


In [ ]:
gender_counts = df.groupby("gender").size().rename("count")
age_counts = df.groupby("age").size().rename("count")
gender_age_counts = df.groupby(["gender", "age"]).size().rename("count")

print("Số bản ghi theo gender:")
display(gender_counts.to_frame())

print("Số bản ghi theo age:")
display(age_counts.to_frame())

print("Số bản ghi theo gender và age:")
display(gender_age_counts.to_frame())


## 6. Tạo các cột mới `gender_num`, `pretest`, `posttest`

Bước này giải quyết các ý 7 và 8.

Quy ước:
- `female = 0`
- `male = 1`
- `pretest = (pretest1 + pretest2 + pretest3) / 3`
- `posttest = (posttest1 + postest2 + postest3) / 3`


In [ ]:
df["gender_num"] = df["gender"].apply(
    lambda value: 1 if str(value).strip().lower() == "male" else 0
)

df["pretest"] = df[["pretest1", "pretest2", "pretest3"]].mean(axis=1)
df["posttest"] = df[["posttest1", "postest2", "postest3"]].mean(axis=1)

display(df.head())


## 7. Mean, median, mode và chia nhóm theo percentile

Bước này giải quyết các ý 9 và 10.

Ta chia `pretest` theo các mốc:
- 0%
- 25%
- 50%
- 75%
- 100%

Sau đó gán nhãn:
- `Low`
- `Medium`
- `Good`
- `Excellent`


In [ ]:
pretest_mean = df["pretest"].mean()
pretest_median = df["pretest"].median()
pretest_mode = df["pretest"].mode().iloc[0]

print(f"Mean   : {pretest_mean:.4f}")
print(f"Median : {pretest_median:.4f}")
print(f"Mode   : {pretest_mode:.4f}")

pretest_bins = df["pretest"].quantile([0.00, 0.25, 0.50, 0.75, 1.00]).tolist()
pretest_bins[0] -= 1e-9
pretest_bins[-1] += 1e-9

df["pretest_level"] = pd.cut(
    df["pretest"],
    bins=pretest_bins,
    labels=LABELS,
    include_lowest=True,
)

print("\nCác khoảng phân nhóm của pretest:")
for left, right, label in zip(pretest_bins[:-1], pretest_bins[1:], LABELS):
    print(f"{label:<10} -> ({left:.4f}, {right:.4f}]")

print("\nSố lượng từng nhóm:")
display(df["pretest_level"].value_counts().sort_index().to_frame("count"))


## 8. Ma trận tương quan và giải thích mối quan hệ với `pretest`

Bước này giải quyết ý 11.


In [ ]:
corr_matrix = df.select_dtypes(include=np.number).corr()
corr_with_pretest = corr_matrix["pretest"].sort_values(ascending=False)

print("Tương quan với pretest:")
display(corr_with_pretest.to_frame("correlation"))

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
save_fig("02_correlation_heatmap.png")

print("Nhận xét:")
print("- pretest tương quan mạnh với pretest1, pretest2, pretest3 vì nó được tạo trực tiếp từ 3 cột này.")
print("- Với các biến không dẫn xuất, grade và hours là hai biến có tương quan dương cao hơn các biến còn lại, nhưng mức tương quan vẫn rất yếu.")


## 9. Trực quan phân phối của `pretest`

Bước này giải quyết ý 12.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["pretest"], bins=15, kde=True, ax=axes[0], color="#264653")
axes[0].set_title("Histogram + KDE của pretest")

sns.violinplot(y=df["pretest"], ax=axes[1], color="#e9c46a")
axes[1].set_title("Violin plot của pretest")

save_fig("03_pretest_distribution.png")


## 10. Heatmap và scatter plot cho `pretest`

Bước này giải quyết ý 13.


In [ ]:
pivot_table = df.pivot_table(
    values="pretest",
    index="age",
    columns="gender",
    aggfunc="mean",
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.heatmap(pivot_table, annot=True, fmt=".1f", cmap="YlGnBu", ax=axes[0])
axes[0].set_title("Mean pretest theo age và gender")

sns.scatterplot(
    data=df,
    x="grade",
    y="pretest",
    hue="gender",
    size="hours",
    palette="Set2",
    ax=axes[1],
)
axes[1].set_title("Scatter plot: grade vs pretest")

save_fig("04_pretest_heatmap_and_scatter.png")


## 11. Boxplot của `pretest` theo gender hoặc age

Bước này giải quyết ý 14.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.boxplot(data=df, x="gender", y="pretest", hue="gender", dodge=False, ax=axes[0])
axes[0].set_title("Boxplot pretest theo gender")
legend = axes[0].get_legend()
if legend is not None:
    legend.remove()

sns.boxplot(data=df, x="age", y="pretest", hue="gender", ax=axes[1])
axes[1].set_title("Boxplot pretest theo age và gender")

save_fig("05_pretest_boxplots.png")


## 12. Vẽ đồ thị 2D và 3D cho `pretest`

Bước này giải quyết ý 15.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.regplot(data=df, x="grade", y="pretest", scatter_kws={"alpha": 0.65}, ax=axes[0])
axes[0].set_title("2D: grade vs pretest")

sns.regplot(data=df, x="hours", y="pretest", scatter_kws={"alpha": 0.65}, ax=axes[1])
axes[1].set_title("2D: hours vs pretest")

save_fig("06_pretest_2d_plots.png")

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
scatter = ax.scatter(
    df["age"],
    df["hours"],
    df["pretest"],
    c=df["grade"],
    cmap="viridis",
    s=60,
    alpha=0.8,
)
ax.set_xlabel("age")
ax.set_ylabel("hours")
ax.set_zlabel("pretest")
ax.set_title("3D: age, hours và pretest")
fig.colorbar(scatter, ax=ax, shrink=0.6, label="grade")

save_fig("07_pretest_3d_plot.png")


## 13. Xác định các yếu tố ảnh hưởng nhiều nhất đến `pretest`

Bước này giải quyết ý 16.

Ở đây ta chỉ xét các biến đầu vào thực tế hơn:
- `gender_num`
- `age`
- `exercise`
- `hours`
- `grade`


In [ ]:
influential_factors = (
    df[["gender_num", "age", "exercise", "hours", "grade", "pretest"]]
    .corr(numeric_only=True)["pretest"]
    .drop("pretest")
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

display(influential_factors.to_frame("correlation_with_pretest"))

print("Kết luận:")
print("- grade và hours là hai biến ảnh hưởng tương đối cao hơn các biến còn lại.")
print("- Tuy nhiên mọi hệ số đều nhỏ, nên ảnh hưởng thực tế không mạnh.")


## 14. Xây dựng mô hình dự đoán `pretest`

Bước này giải quyết các ý 17, 18 và 19.

Mô hình sử dụng:
- Biến đầu vào: `gender`, `age`, `exercise`, `hours`, `grade`
- Biến mục tiêu: `pretest`
- Mô hình: `Ridge Regression`


In [ ]:
feature_columns = ["gender", "age", "exercise", "hours", "grade"]
target_column = "pretest"

X = df[feature_columns]
y = df[target_column]

numeric_features = ["age", "exercise", "hours", "grade"]
categorical_features = ["gender"]

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ]
)

model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("regressor", Ridge()),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print(f"R2 Score : {r2:.4f}")
print(f"RMSE     : {rmse:.4f}")


### Trực quan kết quả mô hình


In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.75, color="#457b9d")

line_start = min(y_test.min(), y_pred.min())
line_end = max(y_test.max(), y_pred.max())
plt.plot([line_start, line_end], [line_start, line_end], "r--")

plt.xlabel("Actual pretest")
plt.ylabel("Predicted pretest")
plt.title("Actual vs Predicted pretest")
save_fig("08_model_actual_vs_predicted.png")

feature_names = model.named_steps["preprocess"].get_feature_names_out()
coefficients = pd.Series(
    model.named_steps["regressor"].coef_,
    index=feature_names,
).sort_values(key=lambda s: s.abs(), ascending=False)

display(coefficients.to_frame("coefficient"))

plt.figure(figsize=(8, 5))
sns.barplot(
    x=coefficients.values,
    y=coefficients.index,
    orient="h",
    color="#f4a261",
)
plt.title("Model Coefficients")
plt.xlabel("Coefficient value")
save_fig("09_model_coefficients.png")

print("Nhận xét:")
print("- R2 âm nghĩa là mô hình dự đoán còn kém hơn cả cách dự đoán bằng giá trị trung bình.")
print("- RMSE khoảng 12 điểm cho thấy sai số dự đoán còn khá lớn.")
print("- Dữ liệu hiện tại chưa có biến đầu vào đủ mạnh để dự đoán tốt pretest.")


## 15. Confusion Matrix giữa `pretest` và `posttest`

Bước này giải quyết ý 20.

Ta dùng lại các mốc percentile của `pretest`, sau đó áp vào cả `pretest` và `posttest`
để so sánh hai kết quả phân loại.


In [ ]:
cm_edges = pretest_bins.copy()
cm_edges[0] = min(df["pretest"].min(), df["posttest"].min()) - 1e-9
cm_edges[-1] = max(df["pretest"].max(), df["posttest"].max()) + 1e-9

df["posttest_level"] = pd.cut(
    df["posttest"],
    bins=cm_edges,
    labels=LABELS,
    include_lowest=True,
)

cm = confusion_matrix(
    df["pretest_level"].astype(str),
    df["posttest_level"].astype(str),
    labels=LABELS,
)

cm_df = pd.DataFrame(cm, index=LABELS, columns=LABELS)
display(cm_df)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Posttest level")
plt.ylabel("Pretest level")
plt.title("Confusion Matrix: pretest vs posttest")
save_fig("10_confusion_matrix_pretest_posttest.png")

print("Nhận xét:")
print("- Các giá trị không tập trung mạnh trên đường chéo chính.")
print("- Điều này cho thấy nhóm phân loại của posttest không khớp chặt với pretest.")


## 16. Lưu dữ liệu đã xử lý và kết luận ngắn

Kết quả chính:
- Dataset có `249` dòng và `14` cột gốc
- Sau khi tạo thêm biến, ta có `gender_num`, `pretest`, `posttest`, `pretest_level`, `posttest_level`
- `grade` và `hours` có liên hệ dương với `pretest`, nhưng mức liên hệ yếu
- Mô hình Ridge cho kết quả `R²` âm, cho thấy dữ liệu đầu vào chưa đủ mạnh để dự đoán tốt `pretest`

Tất cả biểu đồ được lưu trong thư mục `exam_no1_notebook_output/plots`.


In [ ]:
output_file = OUTPUT_DIR / "prepared_dataset.csv"
df.to_csv(output_file, index=False)

print("Đã lưu dữ liệu đã xử lý vào:", output_file)
print("Đã lưu các biểu đồ vào:", PLOT_DIR)
